In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("all_stock_ratios_df.csv")
display(df.head())

,fiscalDateEnding,Ticker,net_income_minus_taxes_over_net_income,pretax_non_operating_over_sales,sales_over_assets,income_to_equity,earnings_over_price,relative_price_evolution,net_income_to_common_over_net_income_smoothed,liabilities_over_equity,inventory_over_current_liabilities,working_capital_over_sales,dividend_per_share
0,2017-03-01,LECO,0.605114,-0.006173,0.282995,0.071218,0.193118,0.031469,0.0,1.617796,0.666054,1.249459,22.900740
1,2017-04-01,LECO,0.421442,-0.006505,0.301234,0.043143,0.114678,0.057238,0.0,1.537092,0.657183,1.243505,21.446891
2,2017-05-01,LECO,0.400729,-0.006357,0.303231,0.048669,0.132847,0.061394,0.0,1.498753,0.671808,1.244453,21.491741
3,2017-06-01,LECO,0.409378,-0.005864,0.295455,0.072028,0.199637,0.093576,0.0,1.490881,0.696836,1.249215,22.608214
4,2017-07-01,LECO,0.586231,-0.005177,0.285221,0.097321,0.297908,0.036219,0.0,1.502741,0.717163,1.262544,24.315648


In [ ]:
import numpy as np
import pandas as pd



# List of problematic ratios to remove entirely
ratios_to_remove = ['net_income_minus_taxes_over_net_income', 'pretax_non_operating_over_sales', 'dividend_per_share', 'earnings_over_price', 'relative_price_evolution', 'net_income_to_common_over_net_income_smoothed','inventory_over_current_liabilities']

# Keep remaining ratios
ratio_columns = [col for col in df.columns
                 if col not in ['fiscalDateEnding', 'Ticker'] + ratios_to_remove]

print(f"Keeping {len(ratio_columns)} ratios: {ratio_columns}")


for col in ratio_columns:
    if df[col].dtype in ['float64', 'float32']:
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            print(f"WARNING: {col} has {inf_count} infinite values")

Keeping 4 ratios: ['sales_over_assets', 'income_to_equity', 'liabilities_over_equity', 'working_capital_over_sales']


In [ ]:
for col in ratio_columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)


df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='ffill')
df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='bfill')


for col in ratio_columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
from scipy.stats.mstats import winsorize

for col in ratio_columns:
    df[col] = winsorize(df[col], limits=(0.01, 0.01))

In [ ]:
num_unique_tickers = len(set(df["Ticker"]))
print(f"Number of unique stock tickers: {num_unique_tickers}")

Number of unique stock tickers: 182


In [ ]:
X_numpy = []
tickers = []

for ticker, group in df.groupby('Ticker'):
    group_sorted = group.sort_values('fiscalDateEnding')
    ratio_values = group_sorted[ratio_columns].to_numpy()

    # Skip stocks with too few periods
    if len(ratio_values) < 12:
        continue

    X_numpy.append(ratio_values)
    tickers.append(ticker)


if X_numpy:
    from collections import Counter
    shapes = [arr.shape for arr in X_numpy]
    shape_counts = Counter(shapes)
    most_common_shape = shape_counts.most_common(1)[0][0]


    X_numpy_filtered = []
    tickers_filtered = []
    for i, arr in enumerate(X_numpy):
        if arr.shape == most_common_shape:
            X_numpy_filtered.append(arr)
            tickers_filtered.append(tickers[i])

    X_multivariate = np.stack(X_numpy_filtered)
    tickers = tickers_filtered
else:
    X_multivariate = np.array([])
    tickers = []

print(f"Shape: {X_multivariate.shape}")

Shape: (172, 58, 4)


In [ ]:
from sklearn.preprocessing import StandardScaler


X_multivariate[np.isinf(X_multivariate)] = np.nan


if np.isnan(X_multivariate).any():
    print("Warning: NaN values found in X_multivariate before scaling. Filling with 0.0.")
    X_multivariate = np.nan_to_num(X_multivariate, nan=0.0)

print(f"Number of infinite values in X_multivariate after cleaning: {np.isinf(X_multivariate).sum()}")
print(f"Number of NaN values in X_multivariate after cleaning: {np.isnan(X_multivariate).sum()}")

n_stocks, n_periods, n_ratios = X_multivariate.shape
X_2d = X_multivariate.reshape(-1, n_ratios)
scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)
X_scaled = X_2d_scaled.reshape(n_stocks, n_periods, n_ratios)

# Check for outliers again
stock_stds = X_scaled.std(axis=(1, 2))
print(f"Std dev range: {stock_stds.min():.2f} to {stock_stds.max():.2f}")
print(f"Stocks with std > 2.5: {(stock_stds > 2.5).sum()}")

Number of infinite values in X_multivariate after cleaning: 0
Number of NaN values in X_multivariate after cleaning: 0
Std dev range: 0.23 to 4.37
Stocks with std > 2.5: 3


In [ ]:

X_multivariate[np.isinf(X_multivariate)] = np.nan


if np.isnan(X_multivariate).any():
    print("Warning: NaN values found in X_multivariate. Filling with 0.")
    X_multivariate = np.nan_to_num(X_multivariate, nan=0.0)

print(f"Number of infinite values in X_multivariate after cleaning: {np.isinf(X_multivariate).sum()}")
print(f"Number of NaN values in X_multivariate after cleaning: {np.isnan(X_multivariate).sum()}")

Number of infinite values in X_multivariate after cleaning: 0
Number of NaN values in X_multivariate after cleaning: 0


In [ ]:
volatile_stocks = ['INSM', 'LBRDA', 'VHC','CMCT']


keep_mask = [t not in volatile_stocks for t in tickers]


X_clean = X_scaled[keep_mask]


tickers_clean = [t for t in tickers if t not in volatile_stocks]

print(f"Removed {len(tickers) - len(tickers_clean)} stocks due to volatility.")
print(f"New shape: {X_clean.shape}")
print(f"New number of tickers: {len(tickers_clean)}")

Removed 4 stocks due to volatility.
New shape: (168, 58, 4)
New number of tickers: 168


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn_extra.cluster import KMedoids
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings('ignore')

def eros_similarity(evals_i, evecs_i, evals_j, evecs_j, n_components=None):

    if n_components is None:
        n_components = len(evals_i)

    weights_i = evals_i[:n_components] / evals_i[:n_components].sum()
    weights_j = evals_j[:n_components] / evals_j[:n_components].sum()

    weights = (weights_i + weights_j) / 2

    similarity = 0
    for k in range(n_components):
        for l in range(n_components):
            dot_product = np.abs(np.dot(evecs_i[k], evecs_j[l]))
            similarity += weights[k] * weights[l] * dot_product

    return similarity

def eros_distance(evals_i, evecs_i, evals_j, evecs_j, n_components=None):

    sim = eros_similarity(evals_i, evecs_i, evals_j, evecs_j, n_components)
    return np.sqrt(2 - 2 * sim)

def compute_eros_distance_matrix(X, n_components=3, verbose=True):

    n_stocks, n_periods, n_ratios = X.shape

    if verbose:
        print(f"Computing PCA for {n_stocks} stocks with {n_periods} periods and {n_ratios} ratios...")

    pca_results = []

    for i in range(n_stocks):
        if verbose and i % 50 == 0:
            print(f"  Processing stock {i+1}/{n_stocks}")

        stock_data = X[i].T

        stock_data = np.nan_to_num(stock_data, nan=0.0, posinf=1.0, neginf=-1.0)

        # Perform PCA
        pca = PCA(n_components=min(n_components, n_ratios, n_periods))
        pca.fit(stock_data)

        evals = pca.explained_variance_
        evecs = pca.components_

        pca_results.append((evals, evecs))

    if verbose:
        print(f"Computing pairwise EROS distances...")

    # Compute distance matrix
    dist_matrix = np.zeros((n_stocks, n_stocks))

    for i in range(n_stocks):
        if verbose and i % 50 == 0:
            print(f"  Computing distances for stock {i+1}/{n_stocks}")

        evals_i, evecs_i = pca_results[i]

        for j in range(i+1, n_stocks):
            evals_j, evecs_j = pca_results[j]

            # Compute EROS distance
            dist = eros_distance(evals_i, evecs_i, evals_j, evecs_j, n_components)

            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist

    if verbose:
        print(f"Done. Distance matrix shape: {dist_matrix.shape}")
        print(f"Distance range: [{dist_matrix.min():.4f}, {dist_matrix.max():.4f}]")

    return dist_matrix

def find_optimal_k_eros(dist_matrix, min_k=2, max_k=10, min_cluster_size=5, verbose=True):

    results = []

    for k in range(min_k, max_k + 1):
        if verbose:
            print(f"Testing k={k}...", end=" ", flush=True)

        kmedoids = KMedoids(n_clusters=k, random_state=42, metric='precomputed')
        labels = kmedoids.fit_predict(dist_matrix)

        # Check cluster sizes
        unique, counts = np.unique(labels, return_counts=True)
        min_size = counts.min()
        max_size = counts.max()

        # Compute silhouette score
        score = silhouette_score(dist_matrix, labels, metric='precomputed')

        results.append({
            'k': k,
            'silhouette': score,
            'sizes': dict(zip(unique, counts)),
            'min_size': min_size,
            'max_size': max_size
        })

        if verbose:
            sizes_str = ', '.join([f"{c}:{s}" for c, s in zip(unique, counts)])
            print(f"silhouette={score:.4f}, sizes=[{sizes_str}]")

    # Find best k
    best = max(results, key=lambda x: x['silhouette'])

    if verbose:
        print(f"\nBest k={best['k']}, silhouette={best['silhouette']:.4f}")
        print(f"   Cluster sizes: {best['sizes']}")

    return results, best

In [ ]:

print(f"Data shape: {X_clean.shape}")

dist_matrix = compute_eros_distance_matrix(X_clean, n_components=3, verbose=True)

results, best = find_optimal_k_eros(dist_matrix, min_k=2, max_k=30, min_cluster_size=5)

best_k = best['k']
kmedoids = KMedoids(n_clusters=best_k, random_state=42, metric='precomputed')
final_labels = kmedoids.fit_predict(dist_matrix)

cluster_df = pd.DataFrame({
    'Ticker': tickers_clean,
    'Cluster': final_labels
})

print("\n" + "="*50)
print("FINAL CLUSTER ASSIGNMENTS")
print("="*50)
print(cluster_df['Cluster'].value_counts().sort_index())

# Save to CSV
cluster_df.to_csv('stock_clusters_eros.csv', index=False)
print("\n✅ Saved to 'stock_clusters_eros.csv'")

Data shape: (168, 58, 4)
Computing PCA for 168 stocks with 58 periods and 4 ratios...
  Processing stock 1/168
  Processing stock 51/168
  Processing stock 101/168
  Processing stock 151/168
Computing pairwise EROS distances...
  Computing distances for stock 1/168
  Computing distances for stock 51/168
  Computing distances for stock 101/168
  Computing distances for stock 151/168
Done. Distance matrix shape: (168, 168)
Distance range: [0.0000, 1.3921]
Testing k=2... silhouette=0.0488, sizes=[0:101, 1:67]
Testing k=3... silhouette=0.0032, sizes=[0:65, 1:30, 2:73]
Testing k=4... silhouette=-0.0303, sizes=[0:43, 1:27, 2:44, 3:54]
Testing k=5... silhouette=-0.0512, sizes=[0:40, 1:19, 2:47, 3:40, 4:22]
Testing k=6... silhouette=-0.0892, sizes=[0:39, 1:15, 2:37, 3:38, 4:17, 5:22]
Testing k=7... silhouette=-0.1320, sizes=[0:38, 1:30, 2:10, 3:11, 4:32, 5:11, 6:36]
Testing k=8... silhouette=-0.1456, sizes=[0:38, 1:30, 2:9, 3:11, 4:28, 5:9, 6:35, 7:8]
Testing k=9... silhouette=-0.1469, sizes=[